[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/04_data_preprocessing/02_feature_scaling.ipynb)

# Bài 02 — Feature Scaling: chuẩn hoá dữ liệu

Feature Scaling là kỹ thuật đưa các feature về **cùng phạm vi** (same range).

Ví dụ trong bài toán dự đoán giá nhà:
- $x_1$ = diện tích (0 – 2000 feet²)
- $x_2$ = số phòng ngủ (0 – 5)

Hai feature chênh lệch nhau quá lớn về phạm vi. Điều này khiến gradient descent hội tụ **lâu và khó hơn**, khiến các model dựa trên **khoảng cách** (KNN, K-Means, SVM) bị feature có thang đo lớn "lấn át", và làm regularization phạt không công bằng giữa các hệ số.

<img src="images/feature-image.png" style="width:45%;height:45%;">

**Nội dung**
1. Rescaling (min-max)
2. Standardization (z-score) và mean normalization — *(đã sửa lỗi công thức/thuật ngữ so với bản gốc)*
3. *(bổ sung)* Robust scaling; biến đổi **hình dạng** phân phối: log, Box-Cox, Yeo-Johnson
4. *(bổ sung)* Model nào cần scaling? Thực nghiệm với gradient descent và KNN
5. *(bổ sung)* Cài đặt from scratch & quy tắc fit trên train

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/04_data_preprocessing

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, PowerTransformer, RobustScaler, StandardScaler, minmax_scale

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Rescaling (min-max scaling)

Phương pháp đơn giản nhất, đưa feature về đoạn $[0, 1]$:

$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

trong đó $x_{min}, x_{max}$ là giá trị nhỏ nhất / lớn nhất của feature **trong tập train**. (Muốn về $[a, b]$: $x'' = a + x'(b - a)$.)

**Ví dụ:** đưa dữ liệu sinh từ phân phối mũ (exponential) về $[0, 1]$:

In [ ]:
original_data = rng.exponential(size=1000)
scaled_data = minmax_scale(original_data)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(original_data, bins=40, color="#ca8a04")
ax[0].set_title("Original data")
ax[1].hist(scaled_data, bins=40)
ax[1].set_title("Scaled data (min-max)")
plt.show()

Chú ý: **hình dạng phân phối không đổi**, chỉ thang đo thay đổi. Min-max rất nhạy với outlier: một giá trị cực lớn ép toàn bộ dữ liệu còn lại vào một khoảng rất hẹp gần 0. Dữ liệu mới có thể nằm **ngoài** $[0, 1]$ nếu vượt min/max của train.

## 2. Standardization và Mean normalization

### 2.1 Standardization (z-score)

Biến đổi để feature có **trung bình 0** và **độ lệch chuẩn 1**:

$$x' = \frac{x - \mu}{\sigma}$$

trong đó $\mu$ là **trung bình** và $\sigma$ là **độ lệch chuẩn** (standard deviation — *không phải* phương sai) của feature trên tập train. Đây là lựa chọn mặc định cho hầu hết model tuyến tính, SVM, neural network, PCA.

### 2.2 Mean normalization

Chỉ khác z-score ở mẫu số:

$$x' = \frac{x - \mu}{x_{max} - x_{min}}$$

Kết quả có **trung bình 0** và nằm trong khoảng $[-1, 1]$ (không phải $[0, 1]$ như bản gốc viết). Đây là cách chuẩn hoá được dùng trong khoá Machine Learning của Andrew Ng.

In [ ]:
x = rng.normal(70, 25, 500)
z = (x - x.mean()) / x.std()
mn = (x - x.mean()) / (x.max() - x.min())
mm = (x - x.min()) / (x.max() - x.min())
for name, v in [("min-max", mm), ("z-score", z), ("mean normalization", mn)]:
    print(f"{name:20s} mean={v.mean():+.3f}  std={v.std():.3f}  min={v.min():+.3f}  max={v.max():+.3f}")

> **Về thuật ngữ:** "normalization" được dùng với nhiều nghĩa khác nhau tuỳ tài liệu — có khi là min-max, có khi là z-score, có khi là chuẩn hoá vector về độ dài 1 (`sklearn.preprocessing.Normalizer` — chuẩn hoá theo **hàng**!). Hãy luôn xem công thức cụ thể.

## 3. Thay đổi thang đo vs thay đổi hình dạng phân phối

- **Scaling** (min-max, z-score, robust): chỉ đổi **phạm vi**, hình dạng phân phối giữ nguyên.
- **Biến đổi phi tuyến** (log, Box-Cox, Yeo-Johnson): đổi **hình dạng** — làm phân phối lệch trở nên gần **phân phối chuẩn** (đối xứng, hình chuông) hơn.

**Nhắc lại phân phối chuẩn** (phân phối Gauss — "hình chuông"): họ phân phối có cùng dạng, chỉ khác vị trí (trung bình $\mu$) và độ rộng (phương sai $\sigma^2$).

<img src="images/image-2.png" style="width:55%;height:55%;">

- **Box-Cox** (chỉ cho $x > 0$): $x^{(\lambda)} = \frac{x^\lambda - 1}{\lambda}$ ($\lambda \ne 0$), $\log x$ ($\lambda = 0$); $\lambda$ được chọn tự động bằng MLE.
- **Yeo-Johnson**: mở rộng Box-Cox cho cả giá trị âm.
- **RobustScaler**: $(x - \text{median}) / IQR$ — không bị outlier ảnh hưởng.

In [ ]:
skewed = rng.exponential(size=1000)
boxcox_data, lam = stats.boxcox(skewed)
yeo = PowerTransformer(method="yeo-johnson").fit_transform(skewed.reshape(-1, 1)).ravel()

fig, ax = plt.subplots(1, 4, figsize=(18, 3.5))
for a, (name, v) in zip(ax, [("Original", skewed), ("log1p", np.log1p(skewed)),
                             (f"Box-Cox (λ={lam:.2f})", boxcox_data), ("Yeo-Johnson", yeo)]):
    a.hist(v, bins=40)
    a.set_title(f"{name}\nskew = {stats.skew(v):.2f}")
plt.show()

In [ ]:
with_outliers = np.r_[rng.normal(50, 10, 200), [400, 500]]
fig, ax = plt.subplots(1, 3, figsize=(16, 3.2))
for a, (name, scaler) in zip(ax, [("MinMax", MinMaxScaler()), ("Standard", StandardScaler()), ("Robust", RobustScaler())]):
    v = scaler.fit_transform(with_outliers.reshape(-1, 1)).ravel()
    a.hist(v[:-2], bins=30)
    a.set_title(f"{name}: 200 điểm thường nằm trong [{v[:-2].min():.2f}, {v[:-2].max():.2f}]")
plt.show()

Với outlier, MinMax ép các điểm bình thường vào khoảng rất hẹp; Robust giữ thang đo hợp lý cho phần lớn dữ liệu.

## 4. Model nào cần scaling? *(bổ sung)*

| Cần scaling | Vì sao | Không cần | Vì sao |
|---|---|---|---|
| KNN, K-Means | dựa trên **khoảng cách** | Decision Tree | chỉ so sánh ngưỡng trên **từng** feature |
| SVM (RBF, linear) | khoảng cách / margin | Random Forest | như trên |
| Linear/Logistic **có regularization** | phạt hệ số công bằng | Gradient Boosting (XGBoost, LightGBM) | như trên |
| Mọi model train bằng **gradient descent** | hội tụ nhanh | Naive Bayes | xét từng feature độc lập |
| PCA | dựa trên **phương sai** | Linear Regression **không** regularization (giải đóng) | nghiệm tương đương (chỉ đổi đơn vị hệ số) |
| Neural network | ổn định huấn luyện | | |

### Thực nghiệm 1 — Gradient descent hội tụ nhanh hơn

In [ ]:
n = 500
area = rng.uniform(30, 300, n)                 # m²
rooms = rng.integers(1, 6, n).astype(float)
y = 0.04 * area + 0.8 * rooms + rng.normal(0, 0.5, n)
X_raw = np.column_stack([area, rooms])


def gd_mse(X, y, lr, n_iters=300):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    losses = []
    for _ in range(n_iters):
        err = Xb @ w - y
        losses.append(np.mean(err**2))
        w -= lr * 2 / len(y) * Xb.T @ err
    return np.array(losses)


X_std = (X_raw - X_raw.mean(0)) / X_raw.std(0)
loss_raw = gd_mse(X_raw, y, lr=1e-5)          # lr phải rất nhỏ, nếu không sẽ phân kỳ
loss_std = gd_mse(X_std, y, lr=0.1)
fig, ax = plt.subplots()
ax.plot(loss_raw, label="chưa scale (lr = 1e-5, lớn hơn là phân kỳ)")
ax.plot(loss_std, label="đã standardize (lr = 0.1)")
ax.set_yscale("log")
ax.set_xlabel("vòng lặp")
ax.set_ylabel("MSE")
ax.legend()
plt.show()
print(f"MSE sau 300 vòng — chưa scale: {loss_raw[-1]:.3f} | đã scale: {loss_std[-1]:.3f}")

### Thực nghiệm 2 — KNN bị feature có thang đo lớn lấn át

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline

from ml_course.datasets import make_customer_churn

churn = make_customer_churn(n=3000, seed=0)
features = ["tenure_months", "monthly_charges", "total_charges", "support_calls"]
Xc, yc = churn[features].to_numpy(float), churn["churn"].to_numpy()
print("Thang đo:", dict(zip(features, Xc.std(0).round(1))))

for name, model in [("KNN không scale", KNeighborsClassifier(15)),
                    ("KNN + StandardScaler", make_pipeline(StandardScaler(), KNeighborsClassifier(15)))]:
    print(f"{name:22s} accuracy (5-fold CV) = {cross_val_score(model, Xc, yc, cv=5).mean():.3f}")

Không scale, khoảng cách gần như chỉ do `total_charges` (độ lệch chuẩn hàng nghìn) quyết định — `support_calls` (feature rất quan trọng!) bị bỏ qua.

## 5. Cài đặt from scratch và quy tắc fit trên train *(bổ sung)*

Package `ml_course.preprocessing` có phiên bản from scratch theo cùng API `fit` / `transform` với scikit-learn (đã được test so với sklearn trong `tests/`):

In [ ]:
from ml_course.preprocessing import MinMaxScalerScratch, RobustScalerScratch, StandardScalerScratch

X_train, X_test = X_raw[:400], X_raw[400:]
for ours, theirs in [(StandardScalerScratch(), StandardScaler()), (MinMaxScalerScratch(), MinMaxScaler()),
                     (RobustScalerScratch(), RobustScaler())]:
    ours.fit(X_train)                        # ✅ học tham số từ TRAIN
    theirs.fit(X_train)
    same = np.allclose(ours.transform(X_test), theirs.transform(X_test))
    print(f"{type(ours).__name__:24s} khớp sklearn trên test: {same}")

Mở `src/ml_course/preprocessing.py` để xem: mỗi scaler chỉ vài dòng, nhưng xử lý cả trường hợp **cột hằng** (std = 0 → chia cho 1).

### Một vài chú ý
- Khi **scaling**: bạn thay đổi **phạm vi** của dữ liệu.
- Khi **biến đổi phân phối** (log, Box-Cox): bạn thay đổi **hình dạng** phân phối.
- Học tham số (min, max, mean, std, λ) **chỉ từ train**; lưu lại scaler cùng model để dùng khi dự đoán.
- Không scale **biến mục tiêu** cho classification; với regression, scale/log target đôi khi giúp ích — nhớ biến đổi ngược khi dự đoán.

## 🧠 Kiểm tra nhanh

1. Sau z-score, trung bình và độ lệch chuẩn của feature bằng bao nhiêu? Mean normalization thì sao?
2. Feature có vài outlier cực lớn — chọn MinMax, Standard hay Robust scaler?
3. Random Forest có cần scaling không? Vì sao?
4. Dữ liệu test có giá trị lớn hơn max của train — min-max cho kết quả nằm ở đâu?

## Tài liệu tham khảo

[1] [Wikipedia — Phân phối chuẩn](https://vi.wikipedia.org/wiki/Ph%C3%A2n_ph%E1%BB%91i_chu%E1%BA%A9n) · [2] [Scaling vs Normalization](https://kharshit.github.io/blog/2018/03/23/scaling-vs-normalization) · [3] scikit-learn — [Importance of feature scaling](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_scaling_importance.html)

---
<sub>Bài gốc: *Feature Normalization* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 3 (MIT License). Đã sửa: σ là độ lệch chuẩn (không phải phương sai), mean normalization cho khoảng [−1, 1], phân biệt scaling và biến đổi phân phối; bổ sung robust scaling, Yeo-Johnson, thực nghiệm GD/KNN, cài đặt from scratch.</sub>